In [12]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pickle
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import xgboost as xgb
import itertools
import numpy as np
import joblib
import plotly.figure_factory as ff
from sklearn.decomposition import PCA

In [1]:
import pandas as pd

def modify_excel_columns(file_path, output_path):
    # List of columns to check
    columns_to_check = [
        "Si_source_Fumed silica", "Si_source_Ludox-AS40", "Si_source_Ludox-SM30", 
        "Si_source_Na2SiO3", "Si_source_TEOS", "Pre-dissolution of Si", 
        "Al_source_Al(OH)3", "Al_source_NaAlO2", "Pre-dissolution of Al"
    ]
    
    try:
        # Load the Excel file
        df = pd.read_csv(file_path)

        # Check if all required columns exist in the dataframe
        for col in columns_to_check:
            if col not in df.columns:
                raise ValueError(f"Column '{col}' not found in the Excel file.")
        
        # Modify the values based on the condition
        for col in columns_to_check:
            df[col] = df[col].apply(lambda x: 1 if x >= 0.6 else 0)

        # Save the modified DataFrame to a new Excel file
        df.to_csv(output_path, index=False)

        print(f"Modified Excel file saved to {output_path}")

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
input_file_path = './R_fill.csv'  # input data
output_file_path = './R_fill_1.csv'  # intermediate output

modify_excel_columns(input_file_path, output_file_path)


Modified Excel file saved to /Users/eolanrew/Desktop/R_fill/R_fill_1.xlsx


In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

# Load the data
file_path = '/Users/eolanrew/Desktop/R_fill/R_fill_1.xlsx'
data = pd.read_excel(file_path)

# Columns for PCA
columns_for_pca = ['Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica', 'Si_source_Ludox-AS40', 
                   'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS', 
                   'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2', 
                   'Pre-dissolution of Al', 'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)', 
                   'Si precursor sizs (nm)']
data_for_pca = data[columns_for_pca]

# Columns to exclude from scaling
exclude_from_scaling = ['Si_source_Fumed silica', 'Si_source_Ludox-AS40', 'Si_source_Ludox-SM30', 
                        'Si_source_Na2SiO3', 'Si_source_TEOS', 'Pre-dissolution of Si', 
                        'Al_source_Al(OH)3', 'Al_source_NaAlO2', 'Pre-dissolution of Al']

# Identify columns to scale
columns_to_scale = [col for col in columns_for_pca if col not in exclude_from_scaling]

# Apply StandardScaler only to the specified columns
scaler = StandardScaler()
data_for_pca[columns_to_scale] = scaler.fit_transform(data_for_pca[columns_to_scale])

# Save the modified DataFrame to an Excel file
output_file_path = '/Users/eolanrew/Desktop/R_fill/scaled_data_pca_fill.xlsx'
data_for_pca.to_excel(output_file_path, index=False)

# Save the scaler model to a file
scaler_file_path = '/Users/eolanrew/Desktop/R_fill/scaler_model.pkl'
joblib.dump(scaler, scaler_file_path)

print(f"Scaled data saved to {output_file_path}")
print(f"Scaler model saved to {scaler_file_path}")


Scaled data saved to /Users/eolanrew/Desktop/R_fill/scaled_data_pca_fill.xlsx
Scaler model saved to /Users/eolanrew/Desktop/R_fill/scaler_model.pkl


/var/folders/8x/dpn3b6yn343g19p96qcctpgj3nptmd/T/ipykernel_1721/1235804822.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_for_pca[columns_to_scale] = scaler.fit_transform(data_for_pca[columns_to_scale])


In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import pickle

# Load the data
file_path = '/Users/eolanrew/Desktop/R_fill/scaled_data_pca_fill.xlsx'
data = pd.read_excel(file_path)

# Columns for PCA
columns_for_pca = ['Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica', 'Si_source_Ludox-AS40', 
                   'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS', 
                   'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2', 
                   'Pre-dissolution of Al', 'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)', 
                   'Si precursor sizs (nm)']
data_for_pca = data[columns_for_pca]

# PCA with 3 components
pca = PCA(n_components=3)
pca_components = pca.fit_transform(data_for_pca)

# Print variance explained by each principal component
print("\nVariance explained by each principal component:")
for i, variance in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {variance:.4f} ({variance*100:.2f}%)")

# Print cumulative variance explained by the first 3 components
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print(f"\nCumulative variance explained by 3 components: {cumulative_variance[2]:.4f} ({cumulative_variance[2]*100:.2f}%)")

# Creating a DataFrame for the PCA components
pca_columns = [f'PCA_Component_{i+1}' for i in range(3)]
pca_df = pd.DataFrame(pca_components, columns=pca_columns)

# Save the resulting DataFrame to Excel
output_file_path = '/Users/eolanrew/Desktop/R_fill/PCA_fill.xlsx'
pca_df.to_excel(output_file_path, index=False)
print(f'\nFile saved successfully at {output_file_path}')

# Save the PCA model
pca_model_path = '/Users/eolanrew/Desktop/R_fill/pca_fill.pkl'
with open(pca_model_path, 'wb') as file:
    pickle.dump(pca, file)
print(f'PCA model saved successfully at {pca_model_path}')



Variance explained by each principal component:
PC1: 0.4330 (43.30%)
PC2: 0.3263 (32.63%)
PC3: 0.2386 (23.86%)

Cumulative variance explained by 3 components: 0.9980 (99.80%)

File saved successfully at /Users/eolanrew/Desktop/R_fill/PCA_fill.xlsx
PCA model saved successfully at /Users/eolanrew/Desktop/R_fill/pca_fill.pkl


In [4]:
import pandas as pd
import pickle

# Function to make predictions and save the updated file
def predict_and_save(model_path, pca_file_path, output_file_path):
    # Load the PCA-reduced data
    df = pd.read_excel(pca_file_path)
    
    # Ensure the necessary PCA columns are present
    required_columns = ['PCA_Component_1', 'PCA_Component_2', 'PCA_Component_3']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    
    # Extract PCA components
    pca_data = df[required_columns].values
    
    # Load the model
    with open(model_path, 'rb') as file:
        model = pickle.load(file)
    
    # Make predictions (assume model outputs probabilities)
    probabilities = model.predict_proba(pca_data)[:, 1]
    
    # Add the probabilities as a new column to the DataFrame
    df['Probability'] = probabilities
    
    # Save the updated DataFrame to a new Excel file
    df.to_excel(output_file_path, index=False)
    print(f"File with probabilities saved to: {output_file_path}")

# Example usage
model_file_path = '/Users/eolanrew/Desktop/Filtered_dataset/xgb_pca.pkl'
pca_file_path = '/Users/eolanrew/Desktop/R_fill/PCA_fill.xlsx'
output_file_path = '/Users/eolanrew/Desktop/R_fill/pca_fill_prob.xlsx'

predict_and_save(model_file_path, pca_file_path, output_file_path)


File with probabilities saved to: /Users/eolanrew/Desktop/R_fill/pca_fill_prob.xlsx


In [5]:
import pandas as pd
import pickle

# Load the PCA model
pca_model_path = '/Users/eolanrew/Desktop/R_fill/pca_fill.pkl'
with open(pca_model_path, 'rb') as file:
    pca = pickle.load(file)

# Load the PCA results
pca_results_path = '/Users/eolanrew/Desktop/R_fill/pca_fill_prob.xlsx'
pca_results = pd.read_excel(pca_results_path)

# Inverse transform the PCA results
pca_components = pca_results[['PCA_Component_1', 'PCA_Component_2', 'PCA_Component_3']].values
reconstructed_data = pca.inverse_transform(pca_components)

# Create a DataFrame for the reconstructed data
columns_for_pca = [
    'Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica',
    'Si_source_Ludox-AS40', 'Si_source_Ludox-SM30',
    'Si_source_Na2SiO3', 'Si_source_TEOS',
    'Pre-dissolution of Si', 'Al_source_Al(OH)3',
    'Al_source_NaAlO2', 'Pre-dissolution of Al',
    'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)',
    'Si precursor sizs (nm)'
]
reconstructed_df = pd.DataFrame(reconstructed_data, columns=columns_for_pca)

# Add the 'Probability' and 'Set' columns to the reconstructed DataFrame
reconstructed_df['Probability'] = pca_results['Probability'].values
#reconstructed_df['Set'] = pca_results['Set'].values

# Save the reconstructed data to Excel
reconstructed_data_path = '/Users/eolanrew/Desktop/R_fill/pca_fill_inv_pca.xlsx'
reconstructed_df.to_excel(reconstructed_data_path, index=False)
print(f'Reconstructed data saved successfully at {reconstructed_data_path}')


Reconstructed data saved successfully at /Users/eolanrew/Desktop/R_fill/pca_fill_inv_pca.xlsx


In [6]:
import pandas as pd
from numpy.linalg import norm

# Load the data
scaled_data = pd.read_excel('/Users/eolanrew/Desktop/New/Filt_stc.xlsx')
filtered_data = pd.read_excel('/Users/eolanrew/Desktop/R_fill/pca_fill_inv_pca.xlsx')

# Select relevant columns
scaled_features = scaled_data[['Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica', 'Si_source_Ludox-AS40', 
                               'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS', 
                               'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2', 
                               'Pre-dissolution of Al', 'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)', 
                               'Si precursor sizs (nm)']]
filtered_features = filtered_data[['Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica', 'Si_source_Ludox-AS40', 
                                   'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS', 
                                   'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2', 
                                   'Pre-dissolution of Al', 'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)', 
                                   'Si precursor sizs (nm)']]

min_distances = []

# Calculate Euclidean distance
for idx, row in scaled_features.iterrows():
    min_euclidean_distance = float('inf')
    min_filtered_row = None  

    for idx2, row2 in filtered_features.iterrows():
        distance = norm(row - row2)  # L2 norm (Euclidean distance)
        if distance < min_euclidean_distance:
            min_euclidean_distance = distance
            min_filtered_row = idx2  
    min_distances.append({
        'scaled_data_row': idx,
        'filtered_data_row': min_filtered_row,
        'min_euclidean_distance': min_euclidean_distance
    })

# Convert to DataFrame
min_distance_df = pd.DataFrame(min_distances)

# Find largest minimum distances
largest_min_distances = min_distance_df.loc[min_distance_df.groupby('filtered_data_row')['min_euclidean_distance'].idxmax()]

# Select rows from filtered data with the largest minimum distances
selected_rows = filtered_data.loc[largest_min_distances['filtered_data_row']].copy()
selected_rows['max_min_euclidean_distance'] = largest_min_distances['min_euclidean_distance'].values

# Save to Excel
output_path = '/Users/eolanrew/Desktop/R_fill/PCA_fill_distance.xlsx'
with pd.ExcelWriter(output_path) as writer:
    selected_rows.to_excel(writer, sheet_name='Largest_Min_Euclidean_Distances', index=False)
    largest_min_distances.to_excel(writer, sheet_name='Distances_Data', index=False)

# Print the DataFrame
print(largest_min_distances)
print(selected_rows)


     scaled_data_row  filtered_data_row  min_euclidean_distance
10                10                  0                1.470464
72                72                  1                2.016179
24                24                  3                1.663145
17                17                  4                1.981700
75                75                  5                2.519902
92                92                  7                7.861801
100              100                  9                5.517303
74                74                 10                1.343348
19                19                 11                1.408173
32                32                 12                1.998256
38                38                 13                2.694119
11                11                 14                2.130016
56                56                 15                2.493457
1                  1                 16                7.170138
91                91                 17 

In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib  # Import joblib for loading the scaler

# Load the reconstructed data
reconstructed_data_path = '/Users/eolanrew/Desktop/R_fill/PCA_fill_distance.xlsx'
reconstructed_data = pd.read_excel(reconstructed_data_path)

# Load the scaler model
scaler_file_path = '/Users/eolanrew/Desktop/R_fill/scaler_model.pkl'
scaler = joblib.load(scaler_file_path)

# Get the feature names used in the scaler
columns_to_scale = scaler.feature_names_in_  # This gets the names of features used in the scaler
#columns_to_scale = [col for col in columns_to_scale if col != 'EMT']

# Inverse transform the scaled features in the reconstructed data
reconstructed_data[columns_to_scale] = scaler.inverse_transform(reconstructed_data[columns_to_scale])


reconstructed_data['Probability'] = reconstructed_data['Probability']
reconstructed_data['distance'] = reconstructed_data['distance']

# Save the final DataFrame with inverse scaled features to an Excel file
scaler_final_path = '/Users/eolanrew/Desktop/R_fill/pca_fill_final.xlsx'
reconstructed_data.to_excel(scaler_final_path, index=False)

print(f"Inverse scaled DataFrame saved to {scaler_final_path}")


Inverse scaled DataFrame saved to /Users/eolanrew/Desktop/R_fill/pca_fill_final.xlsx


In [8]:
import pandas as pd
import os

def filter_rows_by_column_differences(input_excel, sheet_name, save_directory):
    # Define the columns to analyze
    columns_to_check = [
        "Si_source_Fumed silica",
        "Si_source_Ludox-AS40",
        "Si_source_Ludox-SM30",
        "Si_source_Na2SiO3",
        "Si_source_TEOS"
    ]
    
    # Read the Excel file
    try:
        data = pd.read_excel(input_excel, sheet_name=sheet_name)
    except Exception as e:
        print(f"Error reading the Excel file: {e}")
        return
    
    # Check if the necessary columns are present
    for col in columns_to_check:
        if col not in data.columns:
            print(f"Missing column: {col}")
            return
    
    # Calculate the largest and second largest values for each row
    relevant_data = data[columns_to_check]
    largest = relevant_data.max(axis=1)
    second_largest = relevant_data.apply(lambda row: row.nlargest(2).iloc[-1], axis=1)
    
    # Filter rows where (largest - second_largest) >= 0.7
    condition = (largest - second_largest) >= 0.6
    filtered_data = data[condition]
    
    # Save the filtered rows to a new Excel file
    os.makedirs(save_directory, exist_ok=True)
    save_path = os.path.join(save_directory, "pca_fill_final_1.xlsx")
    try:
        filtered_data.to_excel(save_path, index=False)
        print(f"Filtered rows saved to {save_path}")
    except Exception as e:
        print(f"Error saving the Excel file: {e}")

# Example usage
input_excel = "/Users/eolanrew/Desktop/R_fill/pca_fill_final.xlsx"
sheet_name = "Sheet1"
save_directory = "/Users/eolanrew/Desktop/R_fill"

filter_rows_by_column_differences(input_excel, sheet_name, save_directory)


Filtered rows saved to /Users/eolanrew/Desktop/R_fill/pca_fill_final_1.xlsx


In [9]:
import pandas as pd
import os

def filter_rows_by_column_differences(input_excel, sheet_name, save_directory):
    # Define the columns to analyze
    columns_to_check = [
        "Al_source_Al(OH)3",
        "Al_source_NaAlO2"
    ]
    
    # Read the Excel file
    try:
        data = pd.read_excel(input_excel, sheet_name=sheet_name)
    except Exception as e:
        print(f"Error reading the Excel file: {e}")
        return
    
    # Check if the necessary columns are present
    for col in columns_to_check:
        if col not in data.columns:
            print(f"Missing column: {col}")
            return
    
    # Calculate the largest and second largest values for each row
    relevant_data = data[columns_to_check]
    largest = relevant_data.max(axis=1)
    second_largest = relevant_data.apply(lambda row: row.nlargest(2).iloc[-1], axis=1)
    
    # Filter rows where (largest - second_largest) >= 0.7
    condition = (largest - second_largest) >= 0.6
    filtered_data = data[condition]
    
    # Save the filtered rows to a new Excel file
    os.makedirs(save_directory, exist_ok=True)
    save_path = os.path.join(save_directory, "pca_fill_final_2.xlsx")
    try:
        filtered_data.to_excel(save_path, index=False)
        print(f"Filtered rows saved to {save_path}")
    except Exception as e:
        print(f"Error saving the Excel file: {e}")

# Example usage
input_excel = "/Users/eolanrew/Desktop/R_fill/pca_fill_final_1.xlsx"
sheet_name = "Sheet1"
save_directory = "/Users/eolanrew/Desktop/R_fill"

filter_rows_by_column_differences(input_excel, sheet_name, save_directory)


Filtered rows saved to /Users/eolanrew/Desktop/R_fill/pca_fill_final_2.xlsx
